# TP2 : Préparation de données - Nettoyage de données sales (exercice)
## Dataset : Adult Census Income (version dégradée)

**Contexte :** Les données réelles sont rarement propres. Ce dataset contient les problèmes les plus courants rencontrés en pratique. L'objectif est de partir de `adults_dirty.csv` et d'aboutir à une base propre, documentée, prête pour la modélisation.

**Datamap (dictionnaire des données) :**

| Colonne | Type | Description |
|---|---|---|
| `workclass` | catégorielle | Secteur d'emploi, avec valeurs manquantes |
| `race` | catégorielle | Origine ethnique déclarée |
| `sex` | catégorielle | Sexe, encodé de façon incohérente (Male/male/M/MALE...) |
| `age` | numérique | Âge en années, avec valeurs aberrantes (négatives ou > 100) |
| `education_num` | numérique | Niveau d'éducation encodé (nombre d'années d'études) |
| `hours_per_week` | numérique | Heures travaillées par semaine, avec valeurs aberrantes (> 168) |
| `capital_gain` | numérique | Plus-value en capital, parfois stockée en texte avec virgule (`"5,013"`) |
| `data_source` | *problématique* | Constante sur toutes les lignes : à supprimer, n'apporte aucune information |

> **Version exercice** : les cellules marquées `# TODO` sont à compléter. Le reste (imports, chargement des données, affichages) est déjà fourni.
> Installe les dépendances une seule fois avec `pip install -r requirements.txt` depuis `cours_ml/todo/` (voir le README de ce dossier). Ce TP s'inspire de `cours_ml/05_preparation_donnees/tp_dataprep_2_sale.ipynb` (même méthode, données différentes et plus volumineuses : 3260 individus contre 388 manchots).

---
## 0. Imports & chargement

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
pd.set_option('display.max_columns', None)


In [ ]:
df = pd.read_csv('adults_dirty.csv')
print(f"Dimensions initiales : {df.shape[0]} lignes × {df.shape[1]} colonnes")
df.head(10)


---
## 1. Diagnostic initial

Avant de corriger, on établit un **état des lieux complet** des problèmes.

In [ ]:
df.info()

In [ ]:
# Aperçu des modalités de chaque colonne texte → révèle les incohérences
for col in df.select_dtypes(include='object').columns:
    vals = df[col].unique()
    print(f"=== {col} ({len(vals)} valeurs uniques) ===")
    print(sorted([repr(v) for v in vals if pd.notna(v)])[:20])
    print()


In [ ]:
# Statistiques numériques → révèle les valeurs aberrantes (min/max impossibles)
df.describe().round(2)


> **Observations du diagnostic :**
> - `sex` a des valeurs incohérentes : `Male`, `male`, `M`, `MALE`, `Female`, `female`, `F`, `FEMALE`, `unknown`, `?`, en plus d'espaces parasites
> - `workclass` et `race` ont des problèmes de casse et d'espaces (`PRIVATE`, `  Private  `, `private`...)
> - `age` a des valeurs impossibles (négatives ou > 120)
> - `hours_per_week` dépasse parfois 168 (nombre d'heures dans une semaine)
> - `capital_gain` est stocké en texte avec des virgules de séparation de milliers (`'5,013'`)
> - `data_source` est constante (aucune valeur informative)
> - il y a des doublons et des valeurs manquantes dans `workclass`

---
## 2. Suppression des doublons

In [ ]:
# TODO : detecter et supprimer les doublons complets
# Indice : df.duplicated().sum() puis df.drop_duplicates().reset_index(drop=True)
n_dup = ...
print(f"Doublons complets détectés : {n_dup}")

df = ...
print(f"Après suppression : {df.shape[0]} lignes")


---
## 3. Suppression des variables inutiles

On retire les colonnes sans valeur informative pour l'analyse :
- Colonnes **constantes** (variance nulle)

In [ ]:
# TODO : detecter les colonnes constantes (une seule valeur unique) et les supprimer
# Indice : [c for c in df.columns if df[c].nunique() <= 1], puis df.drop(columns=...)
constant_cols = ...
print(f"Colonnes constantes : {constant_cols}")

df = ...
print(f"Colonnes restantes : {list(df.columns)}")


---
## 4. Correction des erreurs de type

`capital_gain` est en texte avec des virgules (`'5,013'`). On le convertit en numérique.

In [ ]:
print("Avant :", df['capital_gain'].astype(str).unique()[:10])

# TODO : nettoyer capital_gain : retirer les virgules (str.replace) puis convertir en numerique
# Indice : df['capital_gain'].astype(str).str.replace(',', '', regex=False), puis pd.to_numeric(..., errors='coerce')
df['capital_gain'] = ...
df['capital_gain'] = ...

print("Après :", df['capital_gain'].describe())


---
## 5. Recodage et harmonisation des variables catégorielles

On uniformise la casse, on retire les espaces, et on regroupe les modalités équivalentes.

In [ ]:
# Étape 1 : nettoyer espaces et casse pour workclass et race
# TODO : pour chaque colonne, retirer les espaces (str.strip) et uniformiser la casse (str.title)
for col in ['workclass', 'race']:
    df[col] = ...

print("workclass :", sorted(df['workclass'].dropna().unique()))
print("race      :", sorted(df['race'].unique()))


In [ ]:
# Étape 2 : harmoniser sex - regrouper toutes les variantes
print("Avant :", sorted([repr(v) for v in df['sex'].unique()]))

# Dictionnaire de recodage
sex_mapping = {
    'male': 'Male', 'm': 'Male', 'MALE': 'Male', 'Male': 'Male',
    'female': 'Female', 'f': 'Female', 'FEMALE': 'Female', 'Female': 'Female',
}

# TODO : normaliser (strip), remplacer les valeurs manquantes textuelles
# ('unknown', '?', '', 'nan', 'None') par NaN, puis appliquer sex_mapping (insensible a la casse via .str.lower())
df['sex'] = ...
df['sex'] = ...
df['sex'] = df['sex'].apply(lambda x: sex_mapping.get(str(x).lower(), sex_mapping.get(x, x)) if pd.notna(x) else x)

print("Après :", sorted([repr(v) for v in df['sex'].dropna().unique()]))
print(f"NA dans sex : {df['sex'].isna().sum()}")


---
## 6. Détection et traitement des valeurs aberrantes

Deux approches complémentaires :
- **Règles métier** : valeurs physiquement impossibles (âge négatif, > 168h/semaine)
- **Méthode statistique (IQR)** : valeurs au-delà de Q1-1.5×IQR ou Q3+1.5×IQR

In [ ]:
num_cols = ['age', 'education_num', 'hours_per_week', 'capital_gain']

# Boxplots AVANT traitement → les aberrations écrasent l'échelle
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, col in zip(axes, num_cols):
    sns.boxplot(y=df[col], ax=ax, color='lightcoral')
    ax.set_title(f'{col}\n(avant)')
    ax.set_ylabel('')
plt.tight_layout()
plt.show()


In [ ]:
# Étape 1 : règles métier - valeurs impossibles → NaN
# (on les transforme en NaN pour les imputer ensuite, plutôt que de supprimer la ligne)
impossible_rules = {
    'age':             lambda s: (s <= 0) | (s > 100),        # age humain plausible
    'hours_per_week':  lambda s: (s <= 0) | (s > 168),        # heures dans une semaine
}

for col, rule in impossible_rules.items():
    # TODO : appliquer la regle, compter les valeurs aberrantes et les remplacer par NaN
    mask = ...
    n = ...
    if n > 0:
        print(f"{col} : {n} valeurs aberrantes → NaN ({df.loc[mask, col].tolist()[:8]})")
        df.loc[mask, col] = np.nan


In [ ]:
# Étape 2 : méthode IQR pour détecter d'éventuels outliers restants
def detect_outliers_iqr(series):
    # TODO : calculer Q1, Q3, IQR puis les bornes basse/haute (Q1 - 1.5*IQR, Q3 + 1.5*IQR)
    # et renvoyer (masque des outliers, borne basse, borne haute)
    ...

print("Outliers restants selon la règle IQR (1.5×) :")
for col in num_cols:
    mask, low, up = detect_outliers_iqr(df[col].dropna())
    print(f"  {col:20s} : {mask.sum():3d} outliers (bornes : {low:.1f} - {up:.1f})")
# Note : capital_gain a naturellement beaucoup d'outliers IQR (distribution tres asymetrique,
# la plupart des gens ont 0). On les garde : ce sont de vraies valeurs, pas des erreurs.


---
## 7. Imputation des valeurs manquantes

Après nettoyage, on a des NaN (d'origine + aberrations converties). Stratégie :
- **Numériques** : imputation par la **médiane du groupe** (par workclass) - plus précis qu'une médiane globale
- **Catégorielle** : imputation par le **mode**

In [ ]:
# État des NA avant imputation
na_before = df.isnull().sum()
print("Valeurs manquantes avant imputation :")
print(na_before[na_before > 0])


In [ ]:
# TODO : imputer les valeurs manquantes
# Ordre important : imputer 'sex' et 'workclass' par le mode D'ABORD, pour pouvoir ensuite
# grouper par workclass sans lignes orphelines (groupby ignore les groupes NaN par defaut)
for col in ['sex', 'workclass']:
    mode_val = ...
    df[col] = ...
    print(f"{col} imputé par le mode : '{mode_val}'")

# TODO : puis imputer les numeriques par la mediane PAR WORKCLASS (plus precis qu'une mediane globale)
# Indice : df.groupby('workclass')[col].transform(lambda s: s.fillna(s.median()))
for col in num_cols:
    df[col] = ...

print(f"\nValeurs manquantes après imputation : {df.isnull().sum().sum()}")


In [ ]:
# Boxplots APRÈS traitement → échelle redevenue lisible
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, col in zip(axes, num_cols):
    sns.boxplot(y=df[col], ax=ax, color='lightgreen')
    ax.set_title(f'{col}\n(après)')
    ax.set_ylabel('')
plt.tight_layout()
plt.show()


---
## 8. Sélection et filtrage

On peut maintenant **sélectionner les variables d'intérêt** et **filtrer** selon des critères métier.

In [ ]:
# TODO : selectionner les variables pertinentes pour l'analyse
# (race, sex, workclass + les 4 variables numeriques num_cols), copier dans df_final
variables_interet = ...
df_final = ...

# Après suppression de colonnes, de nouveaux doublons peuvent apparaître
n_new_dup = df_final.duplicated().sum()
if n_new_dup > 0:
    print(f'Nouveaux doublons après sélection : {n_new_dup} → supprimés')
    df_final = df_final.drop_duplicates().reset_index(drop=True)

print(f"Variables sélectionnées : {list(df_final.columns)}")

# Exemple de filtrage : ne garder que les individus a temps plein
exemple_filtre = df_final[df_final['hours_per_week'] >= 35]
print(f"\nExemple de filtre (temps plein, >=35h) : {len(exemple_filtre)} individus sur {len(df_final)}")


---
## 9. Validation : le dataset est-il propre ?

In [ ]:
print("=== RAPPORT DE NETTOYAGE ===\n")
print(f"Dimensions finales    : {df_final.shape[0]} lignes × {df_final.shape[1]} colonnes")
print(f"Doublons              : {df_final.duplicated().sum()}")
print(f"Valeurs manquantes    : {df_final.isnull().sum().sum()}")
for col in num_cols:
    print(f"  {col:20s} : {df_final[col].min():.1f} → {df_final[col].max():.1f}")


---
## 10. Analyse descriptive sur données propres

Le nettoyage terminé, on peut refaire l'analyse du TP1 (statistiques, graphiques, corrélations).

### 10.1 Statistiques descriptives

In [ ]:
df_final[num_cols].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col in zip(axes.flat, num_cols):
    ax.hist(df_final[col], bins=25, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(df_final[col].mean(), color='red', linestyle='--', label='Moyenne')
    ax.set_title(col)
    ax.legend(fontsize=8)
plt.suptitle('Distributions après nettoyage', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.flat, num_cols):
    sns.boxplot(data=df_final, x='workclass', y=col, ax=ax, hue='workclass', legend=False)
    ax.set_title(f'{col} par workclass')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()


### 10.2 Corrélations numériques (Pearson)

In [ ]:
corr = df_final[num_cols].corr()
plt.figure(figsize=(7, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, mask=mask,
            linewidths=0.5, square=True, cbar_kws={'label': 'Pearson r'})
plt.title('Corrélations (données nettoyées)')
plt.tight_layout()
plt.show()


### 10.3 V de Cramér (catégorielles)

In [ ]:
def cramers_v(x, y):
    confusion = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion)[0]
    n = confusion.sum().sum()
    phi2 = chi2 / n
    r, k = confusion.shape
    return np.sqrt(phi2 / min(k - 1, r - 1))

cat_cols = ['workclass', 'race', 'sex']
for c1 in cat_cols:
    for c2 in cat_cols:
        if c1 < c2:
            print(f"{c1} / {c2} : V de Cramér = {cramers_v(df_final[c1], df_final[c2]):.3f}")


In [ ]:
# Sauvegarde du dataset propre pour réutilisation
df_final.to_csv('adults_cleaned.csv', index=False)
print(f"Dataset propre sauvegardé : adults_cleaned.csv ({df_final.shape[0]} lignes)")


---
## 11. Synthèse

**Pipeline de nettoyage appliqué (ordre important) :**
1. **Diagnostic** : inspecter types, modalités, statistiques pour repérer les problèmes
2. **Doublons** : `drop_duplicates()`
3. **Variables inutiles** : colonnes constantes
4. **Types** : conversion des nombres stockés en texte (virgules de milliers)
5. **Harmonisation catégorielle** : casse, espaces, regroupement de modalités
6. **Aberrations** : règles métier (impossibilités) + détection IQR
7. **Imputation** : médiane par groupe (numérique), mode (catégoriel)
8. **Sélection / filtrage** : variables d'intérêt et sous-populations (+ re-dédoublonnage)
9. **Validation** : vérifier qu'il ne reste ni doublon, ni NA, ni incohérence

**Bonnes pratiques :**
- Toujours travailler sur une copie, ne jamais modifier le fichier source
- Documenter chaque décision (pourquoi imputer plutôt que supprimer, quel seuil pour les outliers…)
- Revalider systématiquement après chaque étape de nettoyage

---
## Session à rendre

Cette section est à compléter et à rendre à l'issue du TP. Réponds à chaque question dans la
cellule *Réponse* juste en dessous, à partir des résultats que **tu as toi-même obtenus** en
réalisant ce notebook (il n'y a pas de réponse générique valable pour tout le monde : les valeurs
numériques, choix d'hyperparamètres et graphiques dépendent de ton exécution).

**Q1.** Combien de doublons complets as-tu détectés et supprimés en première étape ?

*Réponse :*

_(à compléter)_

**Q2.** Quelle(s) colonne(s) constante(s) as-tu identifiée(s) et supprimée(s) ?

*Réponse :*

_(à compléter)_

**Q3.** Quelle anomalie de format as-tu corrigée sur capital_gain, et pourquoi pandas ne pouvait-il pas la traiter automatiquement comme un nombre ?

*Réponse :*

_(à compléter)_

**Q4.** Combien de valeurs manquantes restait-il juste avant l'étape d'imputation, et sur quelles colonnes ?

*Réponse :*

_(à compléter)_

**Q5.** Pourquoi faut-il impérativement imputer workclass (par le mode) AVANT d'imputer les colonnes numériques par la médiane groupée par workclass ? Qu'est-ce qui se passerait si on inversait l'ordre ?

*Réponse :*

_(à compléter)_

**Q6.** Après nettoyage complet, quelles sont les dimensions finales du dataset, et combien reste-t-il de valeurs manquantes / doublons ?

*Réponse :*

_(à compléter)_